In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import sys
sys.path.append('..')
from src import functions as fc

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

In [26]:
df_modeling = fc.load_data_clean("bank_final.csv")

Buscando archivo en: /Users/jbp/Desktop/IRONHACK/SEMANA7/ML_project/data/cleaned/bank_final.csv


In [27]:
features = df_modeling.drop(columns=["target", "duration"])
target = df_modeling["target"]

In [28]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

In [ ]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)

x_test_scaled = scaler.transform(x_test)

ADA Boosting:

In [30]:
ada_class = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),  n_estimators=200, learning_rate=0.1, 
                               random_state=42)

Al principio cometimos el error de poner max depth en 20. Ada aprende de sus errores y con tanta profundidad no deja error para el aprendizaje. Lo ponemos a 1 (stumpt) para que aprenda mejor. Aumentamos el num de estimators al reducir el max depth. Añadimos un learning rate para mejorar el resultado, al 0.1 para que aprenda lento y mejor. Random state en 42 para que no varíe el resultado en el próximo run.

In [31]:
ada_class.fit(x_train_scaled, y_train)

,estimator,DecisionTreeC...r(max_depth=1)
,n_estimators,200
,learning_rate,0.1
,algorithm,'deprecated'
,random_state,42
,criterion,'gini'
,splitter,'best'
,max_depth,1
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [32]:
pred = ada_class.predict(x_test_scaled)

In [33]:
accuracy5 = ada_class.score(x_test_scaled, y_test)
print(f"La precisión del modelo es: {accuracy5:.2f}")

La precisión del modelo es: 0.65


In [34]:
importancias_ada = pd.DataFrame({
    'feature': features.columns, 
    'importance': ada_class.feature_importances_
}).sort_values('importance', ascending=False)

print(importancias_ada.head(10))

              feature  importance
2               pdays    0.390222
7           euribor3m    0.244048
8         nr.employed    0.228322
42   contact_cellular    0.043527
50          month_may    0.032447
61   poutcome_success    0.025607
45          month_aug    0.018862
43  contact_telephone    0.016965
41           loan_yes    0.000000
44          month_apr    0.000000


FINALMENTE EL ADA BOOST ES EL MÁS ROBUSTO Y CON MEJOR RESULTADOS DESPUÉS DE VARIAR LOS PARÁMETROS EN TODOS LOS MODELOS PROBADOS.

Grid search:

In [43]:
from sklearn.model_selection import GridSearchCV

param_grid_ada = {
    'n_estimators': [50, 100, 200],      
    'learning_rate': [0.01, 0.1, 1.0],   
    'estimator__max_depth': [1, 2, 3]}

grid_ada = GridSearchCV(
    estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier()), 
    param_grid=param_grid_ada,
    cv=5,                  
    scoring='f1',           
    n_jobs=-1,              
    verbose=2 )

grid_ada.fit(x_train, y_train)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=50; total time=   0.2s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=50; total time=   0.2s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=50; total time=   0.2s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=50; total time=   0.2s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=50; total time=   0.2s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=100; total time=   0.3s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=100; total time=   0.3s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=100; total time=   0.3s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=100; total time=   0.3s
[CV] END estimator__max_depth=1, learning_rate=0.01, n_estimators=100; total time=   0.3s
[CV] END estimator__max_depth=1, learning_r

,estimator,AdaBoostClass...eClassifier())
,param_grid,"{'estimator__max_depth': [1, 2, ...], 'learning_rate': [0.01, 0.1, ...], 'n_estimators': [50, 100, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [44]:
grid_ada.best_params_

{'estimator__max_depth': 3, 'learning_rate': 1.0, 'n_estimators': 200}

In [45]:
best_model = grid_ada.best_estimator_

In [46]:
accuracy2 = best_model.score(x_test, y_test)
print(f"La precisión del modelo es: {accuracy2:.2f}")

La precisión del modelo es: 0.63


Random Search:

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform

param_dist_ada = {

    'n_estimators': randint(50, 300),  

    'learning_rate': loguniform(0.01, 1.0),

    'estimator__max_depth': [1, 2, 3],  

    'estimator__min_samples_split': randint(2, 20),

    'estimator__min_samples_leaf': randint(1, 10)}

random_ada = RandomizedSearchCV(

    estimator=AdaBoostClassifier(
        estimator=DecisionTreeClassifier(),
        random_state=42),

    param_distributions=param_dist_ada,
    n_iter=30,     
    cv=5,
    scoring='f1',
    n_jobs=1,        
    verbose=2,
    random_state=42)

random_ada.fit(x_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END estimator__max_depth=3, estimator__min_samples_leaf=4, estimator__min_samples_split=16, learning_rate=0.29106359131330695, n_estimators=238; total time=   1.2s
[CV] END estimator__max_depth=3, estimator__min_samples_leaf=4, estimator__min_samples_split=16, learning_rate=0.29106359131330695, n_estimators=238; total time=   0.9s
[CV] END estimator__max_depth=3, estimator__min_samples_leaf=4, estimator__min_samples_split=16, learning_rate=0.29106359131330695, n_estimators=238; total time=   0.9s
[CV] END estimator__max_depth=3, estimator__min_samples_leaf=4, estimator__min_samples_split=16, learning_rate=0.29106359131330695, n_estimators=238; total time=   0.9s
[CV] END estimator__max_depth=3, estimator__min_samples_leaf=4, estimator__min_samples_split=16, learning_rate=0.29106359131330695, n_estimators=238; total time=   0.9s
[CV] END estimator__max_depth=1, estimator__min_samples_leaf=7, estimator__min_samples_split=

,estimator,AdaBoostClass...ndom_state=42)
,param_distributions,"{'estimator__max_depth': [1, 2, ...], 'estimator__min_samples_leaf': <scipy.stats....t 0x14b114050>, 'estimator__min_samples_split': <scipy.stats....t 0x14aef49e0>, 'learning_rate': <scipy.stats....t 0x149eba780>, ...}"
,n_iter,30
,scoring,'f1'
,n_jobs,1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [61]:
random_ada.best_params_

{'estimator__max_depth': 3,
 'estimator__min_samples_leaf': 2,
 'estimator__min_samples_split': 13,
 'learning_rate': np.float64(0.7568292060167615),
 'n_estimators': 239}

In [62]:
best_model2 = random_ada.best_estimator_

In [70]:
accuracy3 = best_model2.score(x_test, y_test)
print(f"La precisión del modelo es: {accuracy3:.2f}")

La precisión del modelo es: 0.64


In [71]:
from sklearn.metrics import precision_score, recall_score, f1_score

pred3 = best_model2.predict(x_test)

precision3 = precision_score(y_test, pred3)
recall3 = recall_score(y_test, pred3)
f13 = f1_score(y_test, pred3)

print(f"Precision del modelo es: {precision3:.2f}")
print(f"Recall del modelo es: {recall3:.2f}")
print(f"F1-score del modelo es: {f13:.2f}")

Precision del modelo es: 0.58
Recall del modelo es: 0.39
F1-score del modelo es: 0.46
